# week06 · 노트북 01 — LAMMPS 첫 실행: W03 의 LJ 유체 64원자

덱 6-1 · 7장(P6-1). W03 노트북 02 가 파이썬으로 돌린 상자(탄소 ε 0.109 · σ 3.40 Å · L 17.47 Å · 64원자)를 LAMMPS 두 파일(`lj.data` · `in.lj`)로 쓰고 한 줄로 돌린다.

실행: week06 폴더에서 `claude` → 「`@notebooks/01_lammps_first_run.ipynb` 를 실행하고 결과를 요약해」(Claude Code 가 week06 폴더에서 `python ../run_nb.py notebooks/01_lammps_first_run.ipynb` 로 실행) · 또는 JupyterLab 에서 직접. 순서: 1/4 환경 확인 → ▶ 2/4 파일 두 개 만들기 → ▶ 3/4 실행 → ▶ 4/4 thermo 표 읽기. 결과 = `out/log.lj`. LAMMPS 가 없으면 4/4 가 `expected_output/log.lj` 를 읽는다.

In [ ]:
# 1/4 환경 확인 · 기본 = 로컬 mdai(lmp · packmol 포함) · Claude Code 실행 · 아래 if 는 (선택) Colab 에서만 저장소 받기 · LAMMPS 설치
import os, subprocess, sys, pathlib
if 'google.colab' in sys.modules:
    if not pathlib.Path('/content/mooc').exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git','/content/mooc'], check=False)
    os.chdir('/content/mooc/week06')
    # conda-forge 고정 버전 (덱 5장) — 약 3분
    subprocess.run('pip -q install condacolab && python -c "import condacolab; condacolab.install()"', shell=True)
    subprocess.run('conda install -y -q -c conda-forge "lammps=2025.07.22=cpu_py311*nompi*" packmol=21.2.1', shell=True)
else:
    os.chdir(pathlib.Path.cwd() if pathlib.Path('scripts').exists() else pathlib.Path(__file__).resolve().parents[1])
import shutil; LMP = shutil.which('lmp') or shutil.which('lmp_serial'); PACKMOL = shutil.which('packmol')
print('week06 폴더:', os.getcwd()); print('lmp:', LMP, '· packmol:', PACKMOL, '— None 이면 expected_output 의 로그로 진행')


In [ ]:
# ▶ 2/4 lj.data · in.lj 작성 (scripts/build_lj_lammps.py 와 같은 코드 · 덱 7장)
import subprocess, sys
print(subprocess.run([sys.executable, 'scripts/build_lj_lammps.py'], capture_output=True, text=True).stdout)
print(open('out/in.lj').read())

In [ ]:
# ▶ 3/4 실행 — lmp 한 줄 (64원자 · 1,000스텝 · 1 초 안)
import subprocess, pathlib
if LMP:
    r = subprocess.run([LMP, '-in', 'in.lj', '-log', 'log.lj'], cwd='out', capture_output=True, text=True)
    print('exit', r.returncode); print('\n'.join(r.stdout.splitlines()[-4:]))
    LOG = pathlib.Path('out/log.lj')
else:
    LOG = pathlib.Path('expected_output/log.lj'); print('LAMMPS 없음 → expected_output/log.lj 사용')

In [ ]:
# ▶ 4/4 thermo 표 읽기 — E 보존 오차 · Density 고정 · 1 ns 추정 (P6-1 의 확인 포인트)
import re
rows, hdr = [], None
for l in LOG.read_text().splitlines():
    if l.strip().startswith('Step'): hdr = l.split(); continue
    if l.startswith('Loop time'):
        loop = float(l.split()[3]); hdr = None; continue
    if hdr and re.match(r'^\s*\d+\s', l) and len(l.split()) == len(hdr): rows.append(dict(zip(hdr, map(float, l.split()))))
E0, E1 = rows[0]['TotEng'], rows[-1]['TotEng']
print(f"스텝 {int(rows[0]['Step'])} → {int(rows[-1]['Step'])} · T {rows[0]['Temp']:.1f} → {rows[-1]['Temp']:.1f} K")
print(f"TotEng {E0:.3f} → {E1:.3f} kcal/mol · 보존 오차 {abs(E1-E0)/abs(E0)*100:.2f} %  (NVE)")
print(f"Density {rows[0]['Density']:.4f} → {rows[-1]['Density']:.4f} g/cm³ (NVE 는 상자 고정)")
print(f"Loop time {loop:.3f} s / {int(rows[-1]['Step'])} 스텝 → 1 ns(10^6 스텝) ≈ {loop/int(rows[-1]['Step'])*1e6:.0f} s")
# 이 출력과 thermo 표를 P6-1 프롬프트에 붙여 넣는다 (prompts.md)

## 확인
- `Total wall time` 이 찍혔는가 · Density 열이 0.2394 로 일정한가(NVE)
- E 보존 오차 0.3 % 안팎 · 1 ns ≈ 21 s (mdai 1코어)
- 못 돌리면 `expected_output/log.lj` 로 P6-1 진행 → W07